<a href="https://colab.research.google.com/github/EvitaKyprianou/Genomics-portfolio/blob/main/variant_lookup_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
# ============================================================
# SNP Lookup Agent v4 -- Complete final version
# Ensembl + gnomAD + CSV export + summary counts
# ============================================================

import requests
import time
import json
import csv
import os

# ---- Load SNP list from uploaded CSV file ------------------

with open("snp_input.csv", "r") as f:
    reader = csv.reader(f)
    snp_ids = [row[0].strip() for row in reader if row]

print(f"Loaded {len(snp_ids)} variants from snp_input.csv")
print("Starting lookups...\n")

# ---- Configuration -----------------------------------------

ensembl_url = "https://rest.ensembl.org"
ensembl_headers = {"Content-Type": "application/json"}

gnomad_url = "https://gnomad.broadinstitute.org/api"
gnomad_headers = {"Content-Type": "application/json"}

RARE_THRESHOLD = 0.01

# ---- Function 1: Ensembl lookup ----------------------------

def get_ensembl_data(rsid):
    endpoint = f"/variation/human/{rsid}?content-type=application/json"
    url = ensembl_url + endpoint
    response = requests.get(url, headers=ensembl_headers)
    if response.status_code == 200:
        data = response.json()
        mappings = data.get("mappings", [])
        if mappings:
            first = mappings[0]
            return {
                "chromosome":  first.get("seq_region_name", "N/A"),
                "position":    first.get("start", "N/A"),
                "consequence": data.get("most_severe_consequence", "N/A"),
                "alleles":     first.get("allele_string", "N/A")
            }
    return None

# ---- Function 2: gnomAD lookup -----------------------------

def get_gnomad_af(rsid, chromosome, position, alleles):
    if "/" not in str(alleles):
        return None

    parts = alleles.split("/")
    ref = parts[0].strip()
    alt = parts[1].strip()

    variant_ids_to_try = [
        f"{chromosome}-{position}-{ref}-{alt}",
        f"{chromosome}-{position}-{alt}-{ref}"
    ]

    for variant_id in variant_ids_to_try:
        query = """
        {
          variant(variantId: "%s", dataset: gnomad_r4) {
            genome {
              af
            }
          }
        }
        """ % variant_id

        payload = {"query": query}

        try:
            response = requests.post(
                gnomad_url,
                headers=gnomad_headers,
                json=payload,
                timeout=15
            )

            if response.status_code == 200:
                data = response.json()
                variant_data = data.get("data", {}).get("variant")

                if variant_data is None:
                    continue

                genome_data = variant_data.get("genome")
                if genome_data is None:
                    continue

                af = genome_data.get("af")
                if af is not None:
                    return af

        except requests.exceptions.Timeout:
            print(f"  gnomAD timed out for {variant_id}")
            continue
        except requests.exceptions.RequestException as e:
            print(f"  gnomAD request error: {e}")
            continue

    return None

# ---- Function 3: classify allele frequency -----------------

def classify_af(af):
    if af is None:
        return "N/A", "not in gnomAD"
    elif af < RARE_THRESHOLD:
        return f"{af:.6f}", "RARE"
    else:
        return f"{af:.6f}", "common"

# ---- Function 4: export to CSV -----------------------------

def export_to_csv(results, filename="variant_report.csv"):
    fieldnames = [
        "rsID", "Chromosome", "Position", "Alleles",
        "Consequence", "AF_global", "Frequency_class",
        "gnomAD_dataset", "Reference_genome"
    ]
    with open(filename, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow(r)
    abs_path = os.path.abspath(filename)
    print(f"\nCSV saved: {abs_path}")
    print(f"Rows written: {len(results)}")

# ---- Function 5: print final summary -----------------------

def print_summary(results):
    total = len(results)

    rare_variants    = [r for r in results if r["Frequency_class"] == "RARE"]
    common_variants  = [r for r in results if r["Frequency_class"] == "common"]
    error_variants   = [r for r in results if r["Frequency_class"] == "ERROR"]
    no_data_variants = [r for r in results if r["Frequency_class"] == "not in gnomAD"]

    n_rare    = len(rare_variants)
    n_common  = len(common_variants)
    n_error   = len(error_variants)
    n_no_data = len(no_data_variants)

    def pct(n):
        return (n / total * 100) if total > 0 else 0.0

    print("\n" + "="*50)
    print("  FINAL SUMMARY")
    print("="*50)
    print(f"  Total variants queried : {total}")
    print(f"  Common  (AF >= 1%)     : {n_common}  ({pct(n_common):.1f}%)")
    print(f"  Rare    (AF <  1%)     : {n_rare}  ({pct(n_rare):.1f}%)")
    print(f"  Not in gnomAD          : {n_no_data}  ({pct(n_no_data):.1f}%)")
    print(f"  Lookup errors          : {n_error}  ({pct(n_error):.1f}%)")
    print("-"*50)

    if n_rare > 0:
        print("  Rare variants flagged:")
        for r in rare_variants:
            print(f"    {r['rsID']}  chr{r['Chromosome']}:{r['Position']}  AF={r['AF_global']}")
    else:
        print("  No rare variants detected in this batch.")

    print("="*50)
    print(f"  Rare threshold used    : AF < {RARE_THRESHOLD} ({RARE_THRESHOLD*100:.0f}%)")
    print(f"  gnomAD dataset         : gnomad_r4 (v4, GRCh38)")
    print("="*50)

# ---- Main loop ---------------------------------------------

results = []

for rsid in snp_ids:
    print(f"Processing {rsid}...")

    ensembl = get_ensembl_data(rsid)
    time.sleep(1.0)

    if ensembl is None:
        print(f"  Ensembl lookup failed for {rsid}")
        results.append({
            "rsID":             rsid,
            "Chromosome":       "ERROR",
            "Position":         "ERROR",
            "Alleles":          "ERROR",
            "Consequence":      "ERROR",
            "AF_global":        "ERROR",
            "Frequency_class":  "ERROR",
            "gnomAD_dataset":   "gnomad_r4",
            "Reference_genome": "GRCh38"
        })
        continue

    af_raw = get_gnomad_af(
        rsid,
        ensembl["chromosome"],
        ensembl["position"],
        ensembl["alleles"]
    )
    time.sleep(1.0)

    af_str, flag = classify_af(af_raw)

    results.append({
        "rsID":             rsid,
        "Chromosome":       ensembl["chromosome"],
        "Position":         ensembl["position"],
        "Alleles":          ensembl["alleles"],
        "Consequence":      ensembl["consequence"],
        "AF_global":        af_str,
        "Frequency_class":  flag,
        "gnomAD_dataset":   "gnomad_r4",
        "Reference_genome": "GRCh38"
    })

    print(f"  Chr {ensembl['chromosome']}  AF={af_str}  {flag}")

# ---- Print results table -----------------------------------

print("\n" + "="*72)
print("  SNP SUMMARY -- Ensembl + gnomAD v4 (GRCh38)")
print("="*72)
print(f"{'rsID':<12} {'Chr':<4} {'Position':<12} {'AF (global)':<14} {'Flag'}")
print("-"*72)

for r in results:
    print(
        f"{r['rsID']:<12} "
        f"{r['Chromosome']:<4} "
        f"{r['Position']:<12} "
        f"{r['AF_global']:<14} "
        f"{r['Frequency_class']}"
    )

print("="*72)

# ---- Export to CSV -----------------------------------------

export_to_csv(results, filename="variant_report.csv")

# ---- Print final summary -----------------------------------

print_summary(results)

# ---- Download the CSV --------------------------------------

from google.colab import files
files.download("variant_report.csv")

Loaded 6 variants from snp_input.csv
Starting lookups...

Processing rs1800371...
  Chr 17  AF=0.004586  RARE
Processing rs28929474...
  Chr 14  AF=N/A  not in gnomAD
Processing rs4680...
  Chr 22  AF=0.439545  common
Processing rs1805007...
  Chr 16  AF=N/A  not in gnomAD
Processing rs1799971...
  Chr 6  AF=0.129564  common
Processing rs53576...
  Chr 3  AF=0.681997  common

  SNP SUMMARY -- Ensembl + gnomAD v4 (GRCh38)
rsID         Chr  Position     AF (global)    Flag
------------------------------------------------------------------------
rs1800371    17   7676230      0.004586       RARE
rs28929474   14   94378610     N/A            not in gnomAD
rs4680       22   19963748     0.439545       common
rs1805007    16   89919709     N/A            not in gnomAD
rs1799971    6    154039662    0.129564       common
rs53576      3    8762685      0.681997       common

CSV saved: /content/variant_report.csv
Rows written: 6

  FINAL SUMMARY
  Total variants queried : 6
  Common  (AF >= 1%

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>